In [ ]:
'''Architecture of the network
Notebook author: Thuy Duong Ngo
Some implementation details are inspired by code in this repo: <https://github.com/tyui592/A_Learned_Representation_For_Artistic_Style>.
'''

In [ ]:
class StyleTransferNetwork(nn.Module):
    """Style Transfer Network."""

    def __init__(self, num_style=3):
        """Init."""
        super(StyleTransferNetwork, self).__init__()
        self.conv1 = ConvWithCIN(num_style,  3, 32, 1, 'relu', 9)
        self.conv2 = ConvWithCIN(num_style, 32, 64, 2, 'relu', 3)
        self.conv3 = ConvWithCIN(num_style, 64, 128, 2, 'relu', 3)

        self.residual1 = ResidualBlock(num_style, 128, 128)
        self.residual2 = ResidualBlock(num_style, 128, 128)
        self.residual3 = ResidualBlock(num_style, 128, 128)
        self.residual4 = ResidualBlock(num_style, 128, 128)
        self.residual5 = ResidualBlock(num_style, 128, 128)

        self.upsampling1 = UpsamleBlock(num_style, 128, 64)
        self.upsampling2 = UpsamleBlock(num_style, 64, 32)

        self.conv4 = ConvWithCIN(num_style, 32, 3, 1, 'linear', 9)

        """
        init_style_weight = torch.tensor(10.0)
        init_content_weight = torch.tensor(1.0)
        init_tv_weight = torch.tensor(1e-5)

        self.log_style_weight = nn.Parameter(torch.log(init_style_weight))
        self.log_content_weight = nn.Parameter(torch.log(init_content_weight))
        self.log_tv_weight = nn.Parameter(torch.log(init_tv_weight))

        The lines above are implementing style weight and content weight as part of the model.
        That is, they are learnable parameters that can be updated via calling the Adam optimizer
        in the train function. But that shouldn't be it.
        """



    def forward(self, x, style_codes):
        x = self.conv1(x, style_codes)
        x = self.conv2(x, style_codes)
        x = self.conv3(x, style_codes)

        x = self.residual1(x, style_codes)
        x = self.residual2(x, style_codes)
        x = self.residual3(x, style_codes)
        x = self.residual4(x, style_codes)
        x = self.residual5(x, style_codes)

        x = self.upsampling1(x, style_codes)
        x = self.upsampling2(x, style_codes)

        x = self.conv4(x, style_codes)

        return x

class CIN(nn.Module):
    """Conditional Instance Normalization"""

    def __init__(self, num_style, ch):
        super(CIN, self).__init__()
        self.normalize = nn.InstanceNorm2d(ch, affine=False)
        self.offset = nn.Parameter(0.01 * torch.randn(1, num_style, ch))
        self.scale = nn.Parameter(1 + 0.01 * torch.randn(1, num_style, ch))

    def forward(self, x, style_codes):
        b, c, h, w = x.size()
        x = self.normalize(x)
        gamma = torch.sum(self.scale * style_codes, dim=1).view(b, c, 1, 1)
        beta = torch.sum(self.offset * style_codes, dim=1).view(b, c, 1, 1)
        x = x * gamma + beta

        return x.view(b, c, h, w)


class ConvWithCIN(nn.Module):
    """Convolution layer with CIN."""

    def __init__(self, num_style, in_ch, out_ch, stride, activation, ksize):
        super(ConvWithCIN, self).__init__()
        self.padding = nn.ReflectionPad2d(ksize // 2)
        self.conv = nn.Conv2d(in_ch, out_ch, ksize, stride)

        self.cin = CIN(num_style, out_ch)
        if activation == "relu":
            self.activation = nn.ReLU()

        elif activation == "linear":
            self.activation = lambda x: x

    def forward(self, x, style_codes):
        x = self.padding(x)
        x = self.conv(x)
        x = self.cin(x, style_codes)
        x = self.activation(x)

        return x


class ResidualBlock(nn.Module):

    def __init__(self, num_style, in_ch, out_ch):
        super(ResidualBlock, self).__init__()
        self.conv1 = ConvWithCIN(num_style, in_ch, out_ch, 1, "relu", 3)
        self.conv2 = ConvWithCIN(num_style, out_ch, out_ch, 1, "linear", 3)

    def forward(self, x, style_codes):
        out = self.conv1(x, style_codes)
        out = self.conv2(out, style_codes)

        return x + out


class UpsamleBlock(nn.Module):
    def __init__(self, num_style, in_ch, out_ch):
        super(UpsamleBlock, self).__init__()
        self.conv = ConvWithCIN(num_style, in_ch, out_ch, 1, "relu", 3)
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')

    def forward(self, x, style_codes):
        x = self.upsample(x)
        x = self.conv(x, style_codes)

        return x
